# Forecast evaluation & backtesting — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## Evaluate forecasts in the order they would have been made, with metrics that match the cost of being wrong
Forecast evaluation is about honest chronology. These examples compute core errors, compare scale-free metrics, run rolling-origin backtests, expose leakage, and evaluate probabilistic coverage.

In [ ]:
# 1) MAE, RMSE, MAPE on one tiny forecast
y=np.array([10,12,15]); yh=np.array([11,13,14]); e=y-yh; mae=np.mean(np.abs(e)); rmse=math.sqrt(np.mean(e**2)); mape=np.mean(np.abs(e/y))*100
plt.figure(figsize=(6,3)); plt.bar(['MAE','RMSE','MAPE %'],[mae,rmse,mape]); plt.title('three metrics read the same errors differently')
assert abs(mae-1.0)<1e-12 and abs(rmse-1.0)<1e-12 and abs(mape-8.333333333333332)<1e-12

In [ ]:
# 2) MASE scales MAE by a naive forecast baseline
train=np.array([8,10,12,15]); y=np.array([10,12,15]); yh=np.array([11,13,14]); denom=np.mean(np.abs(np.diff(train))); mase=np.mean(np.abs(y-yh))/denom
plt.figure(figsize=(6,3)); plt.bar(['model MAE','naive scale','MASE'],[1.0,denom,mase]); plt.title('MASE < 1 beats naive scale')
assert abs(mase-0.42857142857142855)<1e-12

In [ ]:
# 3) rolling-origin backtest: refit as time moves forward
y=10+0.5*np.arange(20)+np.sin(2*np.pi*np.arange(20)/4); preds=[]; actual=[]
for end in [10,12,14,16]:
    coef=np.polyfit(np.arange(end),y[:end],1); preds.append(np.polyval(coef,end)); actual.append(y[end])
preds=np.array(preds); actual=np.array(actual); err=np.abs(actual-preds)
plt.figure(figsize=(6,3)); plt.plot([10,12,14,16],actual,'o-',label='actual'); plt.plot([10,12,14,16],preds,'s--',label='forecast'); plt.legend(); plt.title('each fold forecasts the future only')
assert len(preds)==4 and err.mean()>0

In [ ]:
# 4) leakage looks good because it trains on the future
t=np.arange(20); y=10+0.4*t+np.sin(2*np.pi*t/5); train_end=12; future=np.arange(train_end,20); coef_ok=np.polyfit(t[:train_end],y[:train_end],1); coef_bad=np.polyfit(t,y,1); mae_ok=np.mean(np.abs(y[future]-np.polyval(coef_ok,future))); mae_bad=np.mean(np.abs(y[future]-np.polyval(coef_bad,future)))
plt.figure(figsize=(6,3)); plt.bar(['proper','leaky'],[mae_ok,mae_bad]); plt.title('future data makes evaluation too optimistic')
assert mae_bad < mae_ok

In [ ]:
# 5) probabilistic backtesting checks coverage of prediction intervals
y=np.array([10,12,11,15,14]); lo=np.array([9,11,10,13,13]); hi=np.array([11,13,12,16,15]); covered=(y>=lo)&(y<=hi); coverage=covered.mean()
plt.figure(figsize=(6,3)); plt.errorbar(range(len(y)),(lo+hi)/2,yerr=[(lo+hi)/2-lo,hi-(lo+hi)/2],fmt='o'); plt.scatter(range(len(y)),y,c=['green' if c else 'red' for c in covered]); plt.title(f'interval coverage = {coverage:.1f}')
assert abs(coverage-1.0)<1e-12